# invoke 的传参
## 1.1 文本的输入


In [ ]:
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
import os
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage,ToolMessage
class LLM:
    def __init__(self):
        env= os.getenv('ENV','local')
        print(f'.env.{env}')
        load_dotenv(f'.env.{env}',override=True)
        
        self.api_key= os.getenv("API_KEY").strip()
        self.model= os.getenv("MODEL").strip()
        self.url= os.getenv("BASE_URL").strip()
        try:
            self.thinking_timeout= float(os.getenv("TINKING_TIMEOUT",30).strip())
            self.num_retries= int(os.getenv("RETRY_NUM",2).strip())
            self.temperature= int(os.getenv("TEMPERATURE",1).strip())
            self.stream= True if os.getenv("STREAM").strip() == 'True' else False
            self.max_token= int(os.getenv("MAX_TOKEN").strip()) if os.getenv("MAX_TOKEN").strip() else None
        except Exception as e:
            raise Exception(f'配置文件传入非法参数。错误信息：{e}')
        self.set_llm()
    def set_llm(self):
        self.llm = ChatDeepSeek(
            model= self.model,
            api_key= self.api_key,
            streaming= self.stream,
            api_base= self.url,
            temperature=self.temperature,
            request_timeout= self.thinking_timeout,
            max_tokens= self.max_token,
            max_retries= self.num_retries,
            # model_kwargs=   {'tools':[]}##用来存放一些langchain没有列出但模型本身支持的，比如tools
            # extra_body= {}  ##基于openai个性化字段 比如thinking
            # configurable_fields= ('model','temperature') ## 用来允许 config中的configurable 覆盖
        )

     

In [10]:
model = LLM().llm
response = model.invoke("只返回两个字符 你好")
print(response)

.env.local
content=[{'type': 'thinking', 'thinking': '我们要求只返回两个字符，即"你好"。但"你好"是两个汉字，每个汉字占两个字节，所以是四个字符？实际上要求的是"两个字符"，但中文通常一个字算一个字符。可能用户希望输出"你好"这两个汉字。直接输出即可。'}, {'type': 'text', 'text': '你好'}] additional_kwargs={'reasoning_content': '我们要求只返回两个字符，即"你好"。但"你好"是两个汉字，每个汉字占两个字节，所以是四个字符？实际上要求的是"两个字符"，但中文通常一个字算一个字符。可能用户希望输出"你好"这两个汉字。直接输出即可。'} response_metadata={'token_usage': Usage(completion_tokens=60, prompt_tokens=10, total_tokens=70, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=58, rejected_prediction_tokens=None, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=10), 'model': 'deepseek/deepseek-v4-flash', 'finish_reason': 'stop', 'logprobs': None, 'model_name': 'deepseek/deepseek-v4-flash', 'model_provider': 'litel

## 1.2 字典列表

In [6]:
model = LLM().llm
messages = [
    {
        'role':'system',
        'content': '你是一个言语简洁的办公助手'
    },
    {
        'role':'user',
        'content':'只返回两个字符 你好'
    }
]
response = model.invoke(messages)
print(response)

.env.local
content='你好' additional_kwargs={'refusal': None, 'reasoning_content': '我们只需要返回两个字符"你好"。但用户指令是"只返回两个字符 你好"，可能意味着输出"你好"两个字。但注意"你好"本身是两个汉字，每个汉字占两个字符（在UTF-8中），但通常我们理解为两个汉字就是两个字符。所以直接输出"你好"即可。'} response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 17, 'total_tokens': 86, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 67, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 17}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '44fe9f61-35b0-431d-9f9a-768f8746ed68', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019fa22c-4c40-7b90-93f8-e53ac291af25-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 17, 'output_tokens': 69, 'total_tokens': 86, 'input_token_details': {'cach

### 1.2.2 涉及多轮对话

In [10]:
model = LLM().llm
messages = [
    {
        'role':'system',
        'content': '你是一个言语简洁的办公助手'
    },
    {
        'role':'user',
        'content':'只返回两个字符 你好'
    }
]
response = model.invoke(messages)
print(response.content)
messages.append({'role':'assistant','content':response.content})
messages.append({'role':'user','content':'我们刚刚要求的是什么'})

response = model.invoke(messages)
print(response.content)

.env.local
你好
要求返回“你好”。


## 1.3 消息列表的输入

In [18]:
model = LLM().llm
messages = [
    SystemMessage(content='你是一个言语简洁的办公助手'),
    HumanMessage(content='只返回两个字符 你好'),
]
response = model.invoke(messages)
print(response.content)
messages.append(AIMessage(content=response.content))
messages.append(HumanMessage(content = '我们刚刚要求的是什么'))
response = model.invoke(messages)
print(response.content)

.env.local
你好
要求只返回两个字符“你好”。


## 1.4 美化输出

In [19]:
from rich import print as rprint

rprint(response)

AIMessage(
    content='要求只返回两个字符“你好”。',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 
'我们刚刚要求的是只返回两个字符"你好"，但用户现在问“我们刚刚要求的是什么”，需要理解用户意图。用户可能是在测试我是否
记得之前的指令和交互。根据对话历史，用户第一次说“你是一个言语简洁的办公助手”，然后说“只返回两个字符 
你好”，我回复了“你好”。现在用户问“我们刚刚要求的是什么”，我需要明确说明：用户要求我作为言语简洁的办公助手，并且只返
回两个字符“你好”。但用户现在的问题本身是一个新的问题，我需要根据要求回答。注意：用户最初设定我是“言语简洁的办公助手
”，所以回答应该简洁。我可以直接回答：要求只返回两个字符“你好”。但为了符合简洁，或许只回复“只返回两个字符‘你好’”或更
短。但是考虑到用户的问题，我需要给出准确答案。同时注意不要超出简洁范围。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 191,
            'prompt_tokens': 26,
            'total_tokens': 217,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 182,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 26
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
        'id': '936d392f-1175-47a9-be90-23b4c6db4848',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019fa241-73f8-7ec1-bceb-cbc5007cf6b1-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 26,
        'output_tokens': 191,
        'total_tokens': 217,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 182}
    }
)

说明：
```
refusal ##触发安全策略
usage_metadata={
        'input_tokens': 26,  ##输入
        'output_tokens': 191,  ##输出
        'total_tokens': 217,
        'input_token_details': {'cache_read': 0},  ##命中缓存
        'output_token_details': {'reasoning': 182}  ##推理token数（算在输出里的）
    }

response_metadata={
        'token_usage': {
            'completion_tokens': 191,
            'prompt_tokens': 26,
            'total_tokens': 217,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,  ##预测性生成数
                'audio_tokens': None,  ##音频token
                'reasoning_tokens': 182,
                'rejected_prediction_tokens': None  ##被拒绝的预测token
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 26
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
        'id': '936d392f-1175-47a9-be90-23b4c6db4848',
        'finish_reason': 'stop',  ##结束原因，正常结束为stop，length是长度超出限制
        'logprobs': None
    },
```

# 流式调用Stream

In [3]:
model = LLM().llm
chunks = model.stream("顺序输出20个质数")
for chunk in chunks :
    print(chunk.text,end = "",flush = True)

.env.local
2
3
5
7
11
13
17
19
23
29
31
37
41
43
47
53
59
61
67
71

# 批量调用 batch


In [4]:
# 一次接收所有响应
messages = [
    "输出你好",
    '输出20个质数',
    '输出我上一个问题'
]
responses = model.batch(messages)
for res in responses:
    print(res)

content='你好！' additional_kwargs={'refusal': None, 'reasoning_content': '好的，用户只说了“输出你好”，这是一个非常简单的指令。用户可能就是想要一个直接的问候回复。不需要任何额外解释或复杂处理，直接输出“你好”作为回应就完全符合要求。'} response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 6, 'total_tokens': 51, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 42, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '0b8f21aa-1507-4a53-afcb-f44e282d0a8e', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019fa277-5c46-74a2-8418-f3ebaacbb99c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 6, 'output_tokens': 45, 'total_tokens': 51, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning'

In [5]:
#按照完成的顺序接收响应
# 一次接收所有响应
messages = [
    "输出你好",
    '输出20个质数',
    '输出我上一个问题'
]
responses = model.batch_as_completed(messages)
for res in responses:
    print(res)

(0, AIMessage(content='你好', additional_kwargs={'refusal': None, 'reasoning_content': '好的，用户只说了“输出你好”，这是一个非常简单的请求，就是让我输出“你好”这两个字。不需要任何额外的解释或处理，直接输出即可。\n\n好的，我直接输出“你好”作为回复。'}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 6, 'total_tokens': 53, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 45, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '224983f0-3405-47fa-acbe-1431f07c110e', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fa279-0eea-7442-8e06-7b89e9128653-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 47, 'total_tokens': 53, 'input_token_details': {'cache_read': 0}, 'output_token

## porfile


In [ ]:
print(model.profile)##很多模型不支持

{}


模型完整参数

In [9]:
from rich import print as rprint
rprint(ChatDeepSeek.model_fields.keys())

dict_keys(['name', 'cache', 'verbose', 'callbacks', 'tags', 'metadata', 'custom_get_token_ids', 'rate_limiter', 
'disable_streaming', 'output_version', 'profile', 'client', 'async_client', 'root_client', 'root_async_client', 
'model_name', 'temperature', 'model_kwargs', 'openai_api_key', 'openai_api_base', 'openai_organization', 
'openai_proxy', 'request_timeout', 'stream_usage', 'max_retries', 'presence_penalty', 'frequency_penalty', 'seed', 
'logprobs', 'top_logprobs', 'logit_bias', 'streaming', 'n', 'top_p', 'max_tokens', 'reasoning_effort', 'reasoning',
'verbosity', 'tiktoken_model_name', 'default_headers', 'default_query', 'http_client', 'http_async_client', 'stop',
'extra_body', 'include_response_headers', 'disabled_params', 'context_management', 'include', 'service_tier', 
'store', 'truncation', 'use_previous_response_id', 'use_responses_api', 'api_key', 'api_base'])